# Enterprise Multimodal RAG: Comprehensive 5-Tier Evaluation Suite

This notebook provides an interactive walkthrough of our **Hierarchical Evaluation Suite** designed for production AI systems:

```text
             ▲
            / \     Level 5: Online Production Monitoring (LangSmith Tracing & Live Feedback)
           /   \    Level 4: Safety & Operational Profiling (Jailbreak, PII, Latency, Cost)
          /     \   Level 3: Application Quality (Correctness, Facet Completeness, Citations)
         /       \  Level 2: Pipeline RAG Triad (Context Relevance, Faithfulness, Answer Relevance)
        /_________\ Level 1: Component Isolation (Retriever & Generator in Isolation)
```

---

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag_pipeline import MultimodalRAGPipeline
from eval.component_eval import ComponentEvaluator
from eval.pipeline_eval import PipelineEvaluator
from eval.application_eval import ApplicationEvaluator
from eval.safety_operational_eval import SafetyOperationalEvaluator

# Initialize pipeline
pipeline = MultimodalRAGPipeline()
print("✅ Pipeline and evaluation modules successfully loaded!")

## 2. Level 1: Component-Level Evaluation (Isolated Testing)

We evaluate the **Retriever** and **Generator** independently to isolate failures.

In [ ]:
comp_eval = ComponentEvaluator(pipeline=pipeline)

# 1. Evaluate Retriever in isolation
ret_results = comp_eval.evaluate_retriever(k_values=[1, 3, 5])
print("--- RETRIEVER SUMMARY ---")
for k, v in ret_results["summary"].items():
    print(f"{k:<30}: {v}")

display(ret_results["details"].head())

# 2. Evaluate Generator with clean, synthetic context (no retrieval noise)
gen_results = comp_eval.evaluate_generator_in_isolation()
print("\n--- GENERATOR ISOLATION SUMMARY ---")
for k, v in gen_results["summary"].items():
    print(f"{k:<30}: {v}")

display(gen_results["details"])

## 3. Level 2: Pipeline-Level Evaluation (The RAG Triad)

Tests the end-to-end synergy across:
1. **Context Relevance**: Did the retriever pull noise-free context for the query?
2. **Faithfulness**: Are all statements grounded in context? (Zero hallucination check)
3. **Answer Relevance**: Did the generator directly answer the question?
4. **Modality Routing Accuracy**: Did visual questions trigger the Vision LLM and text questions trigger the Text LLM?

In [ ]:
pipe_eval = PipelineEvaluator(pipeline=pipeline)
pipe_results = pipe_eval.run_pipeline_evaluation()

print("--- PIPELINE RAG TRIAD SUMMARY ---")
for k, v in pipe_results["summary"].items():
    print(f"{k:<35}: {v}")

display(pipe_results["details"][["ID", "Category", "Routed To", "Routing Match", "Context Relevance (1-5)", "Faithfulness (1-5)", "Answer Relevance (1-5)"]])

## 4. Level 3: Application-Level Evaluation (Task Quality)

Evaluates practical business metrics:
- **Semantic Correctness**: Compared against expert-annotated Ground Truth.
- **Facet Completeness**: Did the response cover 100% of sub-questions in multi-part queries?
- **Citation Accuracy**: Are cited page numbers (`[Page X]`) accurate and grounded?

In [ ]:
app_eval = ApplicationEvaluator(pipeline=pipeline)
app_results = app_eval.run_application_evaluation()

print("--- APPLICATION QUALITY SUMMARY ---")
for k, v in app_results["summary"].items():
    print(f"{k:<35}: {v}")

display(app_results["details"][["ID", "Category", "Correctness (1-5)", "Completeness (%)", "Has Citations", "Citations Valid"]])

## 5. Level 4: Safety & Operational Profiling

Evaluates security defenses and cost/latency profiling:
- **Jailbreak & Prompt Injection Defense**
- **Credential / Secret Protection**
- **Out-of-Domain Fallback Calibration**
- **Latency Profiling (p50, p95) & Cost Economics**

In [ ]:
safe_ops = SafetyOperationalEvaluator(pipeline=pipeline)

# Safety benchmark
safety_res = safe_ops.run_safety_benchmark()
print("--- SAFETY & ADVERSARIAL BENCHMARK ---")
for k, v in safety_res["summary"].items():
    print(f"{k:<35}: {v}")
display(safety_res["details"][["Test ID", "Threat Category", "Risk Type", "Status", "Audit Note"]])

# Operational & Latency profiling
ops_res = safe_ops.run_operational_benchmark()
print("\n--- OPERATIONAL & COST PROFILING ---")
for k, v in ops_res["summary"].items():
    print(f"{k:<35}: {v}")
display(ops_res["details"])

## 6. Level 5: Production Online Evaluation with LangSmith

Test live telemetry streaming and asynchronous online evaluation hooks in LangSmith.

In [ ]:
online_eval = pipeline.online_evaluator
print(f"Project Name        : {online_eval.project_name}")
print(f"LangSmith Connected : {online_eval.is_active}")

# Execute a live query
live_query = "According to the supply-chain diagram, where is the quality lab located?"
live_res = pipeline.query(live_query)

print("\nAnswer:")
print(live_res["answer"])

# Run online judge synchronously for inspection
eval_feedback = online_eval.evaluate_run(
    run_id=live_res.get("run_id"),
    question=live_res["question"],
    answer=live_res["answer"],
    context=live_res["context"],
    routed_to=live_res["routed_to"],
    background=False,
)

print("\nOnline Evaluator Feedback:")
for k, v in eval_feedback.items():
    print(f"  • {k}: {v}")

## 7. Master Executive Scorecard Visualization

In [ ]:
from PIL import Image
chart_path = Path("../assets/full_evaluation_suite.png")
if chart_path.exists():
    display(Image.open(chart_path))
else:
    print("Run 'python eval/run_full_suite.py' to generate the consolidated 4-panel dashboard!")